<div style="text-align:center; border-radius:15px; padding:15px; color:white; margin:0; font-family: 'Orbitron', sans-serif; background: #2E0249; background: #11001C; box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.3); overflow:hidden; margin-bottom: 1em;">  <div style="font-size:150%; color:#FEE100"><b>Fraud Detection Analysis and Prediction</b></div>  <div>This notebook was created with the help of <a href="https://devra.ai/ref/kaggle" style="color:#6666FF">Devra AI</a></div></div>Welcome to this investigative analysis of a fraud detection dataset. Our curiosity is piqued by patterns hidden in transaction data and the challenge of predicting fraudulent behavior. If you find this notebook useful, please consider upvoting it.

# Table of Contents

- [Data Loading and Inspection](#Data-Loading-and-Inspection)
- [Data Cleaning and Preprocessing](#Data-Cleaning-and-Preprocessing)
- [Exploratory Data Analysis (EDA)](#Exploratory-Data-Analysis-EDA)
- [Correlation Analysis](#Correlation-Analysis)
- [Predictive Modeling](#Predictive-Modeling)
- [Conclusions and Future Work](#Conclusions-and-Future-Work)

In [ ]:
# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Importing necessary libraries
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Use Agg backend for matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

# For predictive modeling
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve, auc

# Set some plot styles
sns.set(style='whitegrid', palette='muted', font_scale=1.2)

# Display versions (optional for debugging)
print('numpy version:', np.__version__)
print('pandas version:', pd.__version__)
print('matplotlib version:', matplotlib.__version__)
print('seaborn version:', sns.__version__)


# Data Loading and Inspection

In this section, we load the transaction dataset from the CSV file. The file contains valuable information including the transaction date, amount, transaction type, and whether the transaction is flagged as fraud. We also take a first look at the structure of the data.

In [ ]:
# Load the dataset
df = pd.read_csv('fraud_detection_dataset.csv', encoding='ascii', delimiter=',')

# Display first few rows to inspect the data
print('Shape of the dataset:', df.shape)
print(df.head())

# Data Cleaning and Preprocessing

Now we carry out some data cleaning and preprocessing. Notably, the `Date` column is given as a string, so we convert it to a datetime type. We also verify missing values and check the data types to ensure consistency for further analysis.

In [ ]:
# Convert the Date column to datetime. We infer the date format from the data
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# Check for missing values
missing_values = df.isnull().sum()
print('Missing values in each column:')
print(missing_values)

# Optionally, drop rows with missing Date values as they are crucial for analysis
df = df.dropna(subset=['Date'])

# Convert categorical columns if needed
categorical_cols = ['Transaction_ID', 'Transaction_Type', 'Country', 'Device_Type', 'Payment_Method']
for col in categorical_cols:
    df[col] = df[col].astype('category')

# Ensure numeric columns are of correct type
numeric_cols = ['Amount_USD', 'Account_Age_Months', 'Num_Transactions_Last_24H', 'Is_International', 'Fraud_Label']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print('\nData types after cleaning:')
print(df.dtypes)

# Exploratory Data Analysis (EDA)

Our aim here is to explore insights from the data. We will generate several visualizations including histograms, pie charts, and box plots. Due to the rich numeric data available, we can also examine how these features correlate with each other.

Below, a few visualizations remind us that graphs can speak louder than words, even if sometimes they are a bit shy about revealing secrets.

In [ ]:
# Count plot for Fraud_Label to see distribution of fraudulent vs non-fraudulent transactions
plt.figure(figsize=(8, 4))
sns.countplot(x='Fraud_Label', data=df, palette='Set2')
plt.title('Distribution of Fraudulent Transactions')
plt.xlabel('Fraud Label')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

# Histogram for Amount_USD
plt.figure(figsize=(8, 4))
sns.histplot(df['Amount_USD'], bins=30, kde=True, color='teal')
plt.title('Transaction Amount Distribution')
plt.xlabel('Amount (USD)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

# Box Plot by Transaction Type to inspect Amount_USD across types
plt.figure(figsize=(10, 6))
sns.boxplot(x='Transaction_Type', y='Amount_USD', data=df, palette='Pastel1')
plt.title('Transaction Amount by Type')
plt.xlabel('Transaction Type')
plt.ylabel('Amount (USD)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Pie chart / count plot for Payment Method
plt.figure(figsize=(8, 4))
sns.countplot(x='Payment_Method', data=df, palette='Set3')
plt.title('Count of Payment Methods')
plt.xlabel('Payment Method')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Correlation Analysis

We focus on the numeric columns and check out the correlation matrix. Given that we have at least four numeric columns, a heatmap can efficiently summarize relationships among variables. This helps in understanding clusters and potential predictive factors.

In [ ]:
# Select only numeric columns for correlation analysis
numeric_df = df.select_dtypes(include=[np.number])

# Check if there are at least four numeric columns
if numeric_df.shape[1] >= 4:
    plt.figure(figsize=(10,8))
    correlation_matrix = numeric_df.corr()
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
    plt.title('Correlation Matrix of Numeric Features')
    plt.tight_layout()
    plt.show()
else:
    print('Not enough numeric columns for a proper correlation heatmap.')

# Predictive Modeling

Now that we have a reasonable understanding of the data, we attempt to build a predictor for fraudulent transactions. We use Logistic Regression as a baseline model. We first select a subset of columns as features and split our data into training and test sets. A confusion matrix and ROC curve are then used to evaluate model performance. Note that in real-world applications, further parameter tuning and perhaps more sophisticated models might be necessary.

A historical note: encountering errors with date formats or missing values can be a constant reminder that real data sometimes refuses to fit perfectly into our models. Make sure your preprocessing steps are robust.

In [ ]:
# Feature selection: For simplicity, we will drop Transaction_ID, Date, and other non-numeric features
feature_columns = ['Amount_USD', 'Account_Age_Months', 'Num_Transactions_Last_24H', 'Is_International']

# Ensure there are no missing values in the selected features
data = df.dropna(subset=feature_columns + ['Fraud_Label'])
X = data[feature_columns]
y = data['Fraud_Label']

# Split into train and test sets (70-30 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Initialize and train the Logistic Regression model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Calculate prediction accuracy
accuracy = accuracy_score(y_test, y_pred)
print('Accuracy of the fraud detection predictor:', accuracy)

# Plotting the confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

# Plot ROC Curve
y_pred_prob = model.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label='AUC = {:.2f}'.format(roc_auc))
plt.plot([0, 1], [0, 1], 'r--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

# Conclusions and Future Work

We explored the fraud detection dataset with a combination of visual analytics and predictive modeling. Our approach included a thorough data cleaning process, various exploratory visualizations, and a baseline predictive model using Logistic Regression. 

The strengths of this approach are its systematic data preprocessing and a range of visualizations ranging from histograms to correlation heatmaps, which are useful in identifying key relationships. In future work, one might explore feature engineering (e.g., extracting features from the date column such as day of week or hour of day), try more complex models or ensemble methods, or utilize advanced techniques such as permutation importance to further explain the predictors.

If you find this notebook useful, please consider upvoting it.